# Building LLM Agents Using LangChain & OpenAI API

Complete updated companion, based on the author-supplied LangChain book notebook. Run top to bottom with a locally configured OpenAI key. Calls incur API charges. The career Markdown is an original teaching fixture, not the draft ebook. Original archive code and transcripts remain in the nonexecuting appendix.

## Notebook setup

In [ ]:
import os,sys,subprocess,importlib.metadata,hashlib,urllib.request
from pathlib import Path
from tempfile import TemporaryDirectory
dependencies = {'pandas': '2.3.3', 'numpy': '2.3.3', 'openai': '2.43.0', 'langchain': '1.2.12', 'langchain-openai': '1.1.11', 'langchain-core': '1.4.8', 'langgraph': '1.1.2', 'langchain-community': '0.4.1', 'langchain-text-splitters': '1.1.1', 'ddgs': '9.14.4', 'python-dotenv': '1.2.2', 'pydantic': '2.13.4', 'httpx': '0.28.1', 'certifi': '2025.11.12'}
missing = []
for name,version in dependencies.items():
    try:
        if importlib.metadata.version(name) != version: missing.append(f'{name}=={version}')
    except importlib.metadata.PackageNotFoundError:
        missing.append(f'{name}=={version}')
if missing:
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY locally before running this notebook. Never paste it into a cell.')
os.environ.setdefault('MODEL','gpt-5.4-mini')
os.environ.setdefault('EMBEDDING_MODEL','text-embedding-3-small')
workspace = TemporaryDirectory(prefix='tdb-langchain-12-')
work = Path(workspace.name)
source_revision = '5984620e5691a3991d5e9353bc2cf2533e576234'
files = {'product-reviews.csv': 'ce8895ea5d502554d8da2c7fc6b20e6e9591816ee96e66a8ad329bec7c009522', 'career-in-ai.md': 'bbe0f834b947e46a5e4d152119ee76bb7ed784820d7e057c06c8f1e3b0a3971c'}
for name,digest in files.items():
    url = f'https://raw.githubusercontent.com/youssefHosni/hands-on-langchain/{source_revision}/data/{name}'
    with urllib.request.urlopen(url,timeout=60) as response: payload = response.read()
    assert hashlib.sha256(payload).hexdigest() == digest, name
    (work/name).write_bytes(payload)
print('Python:',sys.version.split()[0])
print('Configured chat model:',os.environ['MODEL'])
print('Pinned example CSV and Markdown inputs verified; no draft ebook is used.')

## Part A — A ReAct agent from scratch

The **ReAct** (Reasoning + Acting) pattern, introduced in the 2022 paper *ReAct: Synergizing Reasoning
and Acting in Language Models*, has the model alternate between **thinking** and **doing**: it reasons
about what it needs, takes an action (calls a tool), observes the result, and repeats until it can
answer. We'll implement the loop ourselves to demystify it.

> Modern models have native tool/function calling that largely supersedes hand-built ReAct prompting for
> production — but building it by hand is the best way to understand what an agent actually does.

### A.1 A minimal agent class

In [ ]:
import os
import re
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))
llm_model = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0)

class ReActAgent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model=llm_model, temperature=0, messages=self.messages,
            max_completion_tokens=2048, reasoning_effort="none",
        )
        return completion.choices[0].message.content

### A.2 The ReAct system prompt

The prompt tells the model to operate in a loop of **Thought → Action → PAUSE → Observation**, lists the
actions it may take, and gives one worked example so it follows the format.

In [ ]:
prompt = """
You operate in a cycle of Thought, Action, PAUSE, and Observation.
At the end of the loop, you provide an Answer.

Use Thought to reason about the information needed to answer the question you were asked.
Use Action to perform one of the available actions, then return PAUSE.
Observation will be the result of performing those actions.

Your available actions are:

get_current_weather:
e.g., get_current_weather: New York
Fetches the current weather for the specified location.

get_forecast:
e.g., get_forecast: San Francisco, 3 days
Provides a weather forecast for the given location and time range.

convert_temperature:
e.g., convert_temperature: 75 F to C
Converts a temperature from Fahrenheit to Celsius or vice versa.

Example session:

Question: What's the current temperature in Paris in Celsius?
Thought: I need to get the current temperature in Paris and convert it to Celsius.
Action: get_current_weather: Paris
PAUSE

You will then be called again with this:

Observation: The current temperature in Paris is 77 F.

You continue:

Thought: I should convert 77 F to Celsius.
Action: convert_temperature: 77 F to C
PAUSE

You will then be called again with this:

Observation: 77 F is approximately 25 C.

You finally output:

Answer: The current temperature in Paris is 25 C.""".strip()

### A.3 The action functions (the agent's tools)

The weather functions below return fixed sample values so we can follow the loop. The temperature converter performs the arithmetic; the later LangGraph example retrieves actual web-search results.

In [ ]:
def get_current_weather(location):
    # Placeholder implementation for fetching weather.
    return f"The current temperature in {location} is 32 F."

def get_forecast(location, days):
    return f"The weather forecast for {location} over the next {days} days is sunny with mild temperatures."

def convert_temperature(conversion):
    try:
        value, unit_from, _, unit_to = conversion.split()
        value = float(value)
        if unit_from.upper() == "F" and unit_to.upper() == "C":
            return f"{value} F is approximately {(value - 32) * 5 / 9:.1f} C."
        elif unit_from.upper() == "C" and unit_to.upper() == "F":
            return f"{value} C is approximately {(value * 9 / 5) + 32:.1f} F."
        return "Invalid temperature conversion. Use 'F to C' or 'C to F'."
    except Exception as e:
        return f"Error in temperature conversion: {e}"

known_actions = {
    "get_current_weather": get_current_weather,
    "get_forecast": get_forecast,
    "convert_temperature": convert_temperature,
}

### A.4 Stepping through the loop by hand

Ask a question and watch the agent's first **Thought** and **Action**:

In [ ]:
abot = ReActAgent(prompt)
result = abot("What is the temperature in Helsinki?")
print(result)

The agent chose an action. We run that tool ourselves and feed the result back as an **Observation**:

In [ ]:
observation = get_current_weather("Helsinki")
print("Observation:", observation)
print()
print(abot(f"Observation: {observation}"))

### A.5 Automating the loop

Doing that by hand is tedious. This `query` function runs the loop automatically: call the model, parse
any `Action`, run the matching tool, feed back the `Observation`, and repeat until the model stops
emitting actions.

In [ ]:
action_re = re.compile(r'^Action: (\w+): (.*)$')   # parse an Action line

def query(question, max_turns=5):
    bot = ReActAgent(prompt)
    next_prompt = question
    for _ in range(max_turns):
        result = bot(next_prompt)
        print(result)
        actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]
        if not actions:
            return result                       # no action -> the agent answered
        action, action_input = actions[0].groups()
        if action not in known_actions:
            raise Exception(f"Unknown action: {action}: {action_input}")
        print(f" -- running {action} {action_input}")
        if action == "get_forecast":
            location, days = [part.strip() for part in action_input.rsplit(",",1)]
            observation = get_forecast(location,days)
        else:
            observation = known_actions[action](action_input)
        print("Observation:", observation)
        next_prompt = f"Observation: {observation}"
    raise RuntimeError("Agent reached max_turns without a final answer")

In [ ]:
query("What is the temperature in Helsinki?")

## Part B — The same agent with LangGraph

Our hand-rolled loop works, but we wrote all the plumbing ourselves. **LangGraph** gives us that loop as
a proper **graph**: nodes do work, edges route control, and a **conditional edge** decides whether to
call a tool or finish. It also handles state and supports modern **native tool calling** (so no more
text parsing) and multiple tool calls in one model response. The action node below executes those calls sequentially.

LangGraph maps our agent to three pieces:
- an **`llm`** node that calls the model,
- a **conditional edge** that checks whether the model requested a tool,
- an **`action`** node that runs the requested tool(s) and loops back to the `llm` node.

### B.1 The search tool and agent state

In [ ]:
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchResults

# DuckDuckGo needs no API key. Return structured results, up to 4 per query.
tool = DuckDuckGoSearchResults(num_results=4, output_format="list")
print(type(tool))
print("tool name:", tool.name)

In [ ]:
# The state is just a list of messages. Annotating with operator.add means new
# messages are APPENDED to the list (not replaced) as the graph runs.
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

### B.2 The agent graph

In [ ]:
from langgraph.graph import StateGraph, START, END

class GraphAgent:
    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_model)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")     # after running tools, go back to the model
        graph.add_edge(START, "llm")        # entry point
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        last = state["messages"][-1]
        return len(last.tool_calls) > 0

    def call_model(self, state: AgentState):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        return {"messages": [self.model.invoke(messages)]}

    def take_action(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t['name']} {t['args']}")
            if t["name"] not in self.tools:
                result = "bad tool name, retry"
            else:
                result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(result)))
        print("Back to the model!")
        return {"messages": results}

In [ ]:
system_prompt = """You are a smart research assistant. Use the search engine to look up information. You are allowed to make multiple calls (either together or in sequence). Only look up information when you are sure of what you want. If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatOpenAI(model=llm_model, temperature=0, max_tokens=2048, max_retries=0, reasoning_effort="none")
abot = GraphAgent(model, [tool], system=system_prompt)

### B.3 Visualizing the graph

LangGraph can render the graph it built. Here is its structure as a Mermaid diagram:

In [ ]:
print(abot.graph.get_graph().draw_mermaid())

### B.4 Running the agent

A single-step question — the agent searches, then answers from what it finds:

In [ ]:
messages = [HumanMessage(content="What is the weather in Helsinki today?")]
result = abot.graph.invoke({"messages": messages}, config={"recursion_limit":12})
print("\n" + result["messages"][-1].content)

A question about **two** cities — modern models can issue **parallel** tool calls, so the agent
can request both searches in one model turn. Our simple action node executes the requested tools in a loop:

In [ ]:
messages = [HumanMessage(content="What is the weather in Helsinki and Stockholm?")]
result = abot.graph.invoke({"messages": messages}, config={"recursion_limit":12})
print("\n" + result["messages"][-1].content)

A **multi-step** question, where later parts depend on earlier answers — the agent searches,
reasons, and searches again as needed:

In [ ]:
query_text = ("Who won the 2022 men's FIFA World Cup, and in which country was it held? "
              "Answer both questions.")
result = abot.graph.invoke({"messages": [HumanMessage(content=query_text)]}, config={"recursion_limit":12})
print("\n" + result["messages"][-1].content)

> **The shortcut.** We built the graph by hand to show how it works. For a maintained higher-level entry point, use `from langchain.agents import create_agent`. Build the graph by hand to understand the loop, then choose the agent interface your application needs.

These examples connect prompts and chains to agents that use tools and maintain a reason/act loop.

## Regression checks

These checks verify data flow and contracts. Explicit scripted probes test control flow only; they are separate from the live model calls above.

In [ ]:
import json,contextlib,io
assert convert_temperature('32 F to C')=='32.0 F is approximately 0.0 C.'
assert convert_temperature('100 C to F')=='100.0 C is approximately 212.0 F.'
assert convert_temperature('0 X to C').startswith('Invalid')
assert convert_temperature('bad input').startswith('Error')
assert '32 F' in get_current_weather('Helsinki')  # explicitly a fixture, not measured weather
assert set(abot.graph.get_graph().nodes)=={'__start__','llm','action','__end__'}
# Explicit scripted control-flow probes: unknown action, two-argument tool dispatch, final return, limit.
real_agent=ReActAgent
try:
    class ScriptedAgent:
        responses=[]
        def __init__(self,*args): self.queue=iter(type(self).responses)
        def __call__(self,message): return next(self.queue)
    ReActAgent=ScriptedAgent
    with contextlib.redirect_stdout(io.StringIO()):
        ScriptedAgent.responses=['Action: get_forecast: Helsinki, 3 days','Answer: Done']
        assert query('test')=='Answer: Done'
        ScriptedAgent.responses=['Action: unknown: x']
        try: query('test')
        except Exception as e: assert 'Unknown action' in str(e)
        else: raise AssertionError('Unknown tool accepted')
        ScriptedAgent.responses=['Action: get_current_weather: Helsinki']
        try: query('test',max_turns=1)
        except RuntimeError: pass
        else: raise AssertionError('Turn limit silently accepted')
finally: ReActAgent=real_agent
assert result['messages'][-1].content
print(json.dumps({'control_flow':'passed','temperature_conversions':'passed','graph_nodes':4,'live_graph_messages':len(result['messages']),'checks':'passed'}))

## Original archive appendix

The following is the complete original article text and all its original code/transcripts, preserved for reference. Fenced examples here are not executed. The maintained tutorial above follows the newer author-supplied notebook.

When we think about large language models (LLM), we often imagine them as super-smart databases filled with internet knowledge, ready to answer any question we throw at them. But the reality is that they are clever assistants, able to understand what we tell them and help us figure things out.

That’s where LangChain’s Agents come in. It’s like giving your assistant superpowers! In agents, a language model is used as a reasoning engine to determine which actions to take and in which order.

In this guide, we’ll dive into what agents are, how to make them, and how to teach them to do all sorts of neat tricks, like searching Wikipedia, solving programming questions, and finally building your own agent.

## Setting Up Working Environment

We will start with setting the environment variables and import the packages and libraries we will use throughout this article.

````python
import os
from langchain.chat_models import ChatOpenAI
from langchain.agents.agent_toolkits import create_python_agent
from langchain.agents import load_tools, initialize_agent
from langchain.agents import AgentType
from langchain.tools.python.tool import PythonREPLTool
from langchain.python import PythonREPL

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
````

Next, we will initialize the LLM and we’re going to use ChatOpenAI . We’re going to set the temperature equal to zero. This is important because we’re going to be using the language model as the reasoning engine of an agent where it’s connecting to other sources of data and computation. Therefore we want this reasoning engine to be as good and as precise as possible. So by setting it to zero, we will get rid of any randomness that might arise.

````python
llm_model = "gpt-3.5-turbo"
llm = ChatOpenAI(temperature=0, model=llm_model)
````

Next, we’re going to load two tools that we’re going to load the LLM math tool and the Wikipedia tool . The LLM math tool is a chain itself, which uses a language model in conjunction with a calculator to do math problems. The Wikipedia tool is an API that connects to Wikipedia, allowing you to run search queries against Wikipedia and get back results.

````python
tools = load_tools(["llm-math","wikipedia"], llm=llm)
````

Next, we are going to initialize the agent with the tools, the language model, and the agent type. Here, we’re going to use the CHAT_ZERO_SHOT_REACT_DESCRIPTION agent type. This is an agent that has been optimized to work with chat models.

We’re also going to pass in handle_parsing_errors equals true. This is useful when the language model might output something that is not able to be parsed into an action and action input, which is the desired output.

When this happens, we’ll pass the misformatted text back to the language model and ask it to correct itself. Finally, we’re going to pass in verbose = true so we can see what is happening step by step.

````python
agent= initialize_agent(
    tools, 
    llm, 
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose = True)
````

## Building Math Tutor Agent

First, we’re going to ask the agent a math question. What is 25% of 300? This is a pretty simple question, but it will be good to understand what exactly is going on.

````python
agent("If you have a rectangular garden with dimensions 12 meters by 8 meters, what is the total area of the garden in square meters?")
````

````text
Entering new AgentExecutor chain…
Question: If you have a rectangular garden with dimensions 12 meters by 8 meters, what is the total area of the garden in square meters?
Thought: We need to calculate the area of the rectangular garden.
Action:
```
{
 “action”: “Calculator”,
 “action_input”: “12*8”
}
```
````

````text
Observation: Answer: 96
Thought: We have the area of the garden, which is 96 square meters.
Final Answer: The total area of the garden is 96 square meters.
````

````text
> Finished chain.
{‘input’: ‘If you have a rectangular garden with dimensions 12 meters by 8 meters, what is the total area of the garden in square meters?’,
 ‘output’: ‘The total area of the garden is 96 square meters.’}
````

So we can see here that when it enters the agent executor chain, it first thinks about what it needs to do. So it has a thought and then has an action. This action is a JSON blob corresponding to two things, an action and an action input .

The action corresponds to the tool to use. So here it says, calculator . The action input is the input to that tool. Here it’s a “12*8”. Next, we can see that there’s an observation that will appear in a separate color if you are running it in the code. The observation , answer equals 96, is coming from the calculator tool itself.

## Building Wikipedia Search Agent

The next agent we are going to explore is using the Wikipedia API . Here, we’re going to ask a question about Andrew NG, and we can look at the intermediate steps to see what it does.

````python
question = "Andrew Ng is a British-American computer scientist \
and the Founder of Coursera and Deep Learning.ai and a Professor at Stanford University\
what books did he write?"
result = agent(question)
````

````text
> Entering new AgentExecutor chain…
Thought: Andrew Ng is a well-known computer scientist and educator. I should use Wikipedia to find out what books he has written.
````

````text
Action:
```
{
 “action”: “Wikipedia”,
 “action_input”: “Andrew Ng”
}
```
````

````text
Observation: Page: Andrew Ng
Summary: Andrew Yan-Tak Ng (Chinese: 吳恩達; born 1976) is a British-American computer scientist and technology entrepreneur focusing on machine learning and artificial intelligence (AI). Ng was a cofounder and head of Google Brain and was the former Chief Scientist at Baidu, building the company’s Artificial Intelligence Group into a team of several thousand people.
Ng is an adjunct professor at Stanford University (formerly associate professor and Director of its Stanford AI Lab or SAIL). Ng has also worked in the field of online education, cofounding Coursera and DeepLearning.AI. He has spearheaded many efforts to “democratize deep learning” teaching over 8 million students through his online courses. Ng is renowned globally in computer science, recognized in Time magazine’s 100 Most Influential People in 2012 and Fast Company’s Most Creative People in 2014. His influence extends to being named in the Time100 AI Most Influential People in 2023.
In 2018, he launched and currently heads the AI Fund, initially a $175-million investment fund for backing artificial intelligence startups. He has founded Landing AI, which provides AI-powered SaaS products.
On April 11, 2024, Amazon announced the appointment of Ng to its board of directors.
````

````text
Page: Coursera
Summary: Coursera Inc. () is a for-profit U.S.-based global massive open online course provider founded in 2012 by Stanford University computer science professors Andrew Ng and Daphne Koller. Coursera works with universities and other organizations to offer online courses, certifications, and degrees in a variety of subjects. 
On February 2, 2021, Coursera announced its B Corporation certification from B Lab and its transformation into a Public Benefit Corporation. 
As of 2023, more than 300 universities and companies were offering courses through Coursera, and by 2024, the number of courses available had risen to approximately 7,000.
````

````text
Page: Andrew Ng (physicist)
Summary: Andrew Kam-hung Ng is a Canadian physicist who is a professor emeritus in the Department of Physics and Astronomy at the Peter Wall Institute for Advanced Studies.
Thought:Based on the information from Wikipedia, it doesn’t seem like Andrew Ng has written any books. However, he has contributed to numerous research papers and articles in the field of machine learning and artificial intelligence.
````

````text
Final Answer: Andrew Ng has not written any books, but has contributed to research papers and articles in the field of machine learning and artificial intelligence.
````

````text
> Finished chain.
````

We can see once again that it thinks and it correctly realizes that it should use Wikipedia . It says action is equal to Wikipedia and action input is equal to Andrew Ng.

The observation that comes back from Wikipedia is three results or three pages, as there are two different Andrew NG and one page for Coursera. We can see the first one covers a computer scientist, and the second one is a physicist.

We can see that the information needed to answer this question, namely the name of the book that he wrote, however, there was no answer for this. This isn’t strictly necessary, and it’s an interesting example to show how agents aren’t perfectly reliable yet.

## Building Python Programming Assistant Agent

The next example we will go through is building a Python Programming Assistant Agent. If you’ve used Copilot or even ChatGPT with the code interpreter plugin enabled, one of the things they’re doing is using the language model to write code and then executing that code and we are going to do this here.

We will start with creating a Python agent and we’re going to use the same LLM as before. We are going to give it a tool, the Python REPL tool. A REPL is a way to interact with code.

So the agent can execute code with this REPL. It will then run, and then we’ll get back some results. And those results will be passed back to the agent so it can decide what to do next.

````python
agent = create_python_agent(
    llm,
    tool=PythonREPLTool(),
    verbose=True
)
````

The problem that we’re going to have this agent solve is we’re going to give it a list of names and then ask it to sort them. So you can see here we have a list of names, Harrison Chase, Lane Chain, LLM, Jeff Fusion, Transformer, GenAI.

````python
employee_list = [["Smith", "John", 35], 
                 ["Doe", "Jane", 28],
                 ["Black", "Michael", 42],
                 ["Brown", "Emily", 31], 
                 ["White", "David", 39], 
                 ["Green", "Sarah", 45],
                 ["Jones", "Christopher", 37]
                ]
````

We will ask the agent to first sort the employees in ascending order regarding their ages and then sort them in descending order regarding their names and then print them.

````python
agent.run(f""" Sort these employees by their age in ascending order and \
then by last name in descending order, and print the output: {employee_list}""")
````

````text
Entering new AgentExecutor chain…

I can use the sorted() function with a lambda function to sort the list by age and then by last name. I can use the reverse parameter to sort the last names in descending order.
Action: Python REPL
Action Input:
```
employees = [[‘Smith’, ‘John’, 35], [‘Doe’, ‘Jane’, 28], [‘Black’, ‘Michael’, 42], [‘Brown’, ‘Emily’, 31], [‘White’, ‘David’, 39], [‘Green’, ‘Sarah’, 45], [‘Jones’, ‘Christopher’, 37]]
sorted_employees = sorted(employees, key=lambda x: (x[2], x[0]), reverse=(False, True))
for employee in sorted_employees:
 print(employee)
```
Observation: TypeError(‘an integer is required (got type tuple)’)
Thought:I made a mistake with the reverse parameter. It should be a boolean value, not a tuple.
Action: Python REPL
Action Input:
```
employees = [[‘Smith’, ‘John’, 35], [‘Doe’, ‘Jane’, 28], [‘Black’, ‘Michael’, 42], [‘Brown’, ‘Emily’, 31], [‘White’, ‘David’, 39], [‘Green’, ‘Sarah’, 45], [‘Jones’, ‘Christopher’, 37]]
sorted_employees = sorted(employees, key=lambda x: (x[2], x[0]), reverse=False)
for employee in sorted_employees:
 print(employee)
```
Observation: [‘Doe’, ‘Jane’, 28]
[‘Brown’, ‘Emily’, 31]
[‘Smith’, ‘John’, 35]
[‘Jones’, ‘Christopher’, 37]
[‘White’, ‘David’, 39]
[‘Black’, ‘Michael’, 42]
[‘Green’, ‘Sarah’, 45]
````

````text
Thought:The employees are now sorted by age in ascending order and then by last name in descending order.
Final Answer: [[‘Doe’, ‘Jane’, 28], [‘Brown’, ‘Emily’, 31], [‘Smith’, ‘John’, 35], [‘Jones’, ‘Christopher’, 37], [‘White’, ‘David’, 39], [‘Black’, ‘Michael’, 42], [‘Green’, ‘Sarah’, 45]]
````

````text
> Finished chain.
“[[‘Doe’, ‘Jane’, 28], [‘Brown’, ‘Emily’, 31], [‘Smith’, ‘John’, 35], [‘Jones’, ‘Christopher’, 37], [‘White’, ‘David’, 39], [‘Black’, ‘Michael’, 42], [‘Green’, ‘Sarah’, 45]]"We can see that when we go into the agent executor chain, it first realizes that it can use the sorted function to list the customers. It’s using a different agent type under the hood, which is why you can see that the action and action input is formatted slightly differently.
````

Here, the action that it takes is to use the Python REPL, and then the action input that you can see is code, where it first writes out employee equals the input list, then sorts the customers, and then goes through this list and prints it.

You can see the agent thinks about what to do and realizes that it needs to write some code. The format that it uses of action and action input is slightly different than before.

It’s using a different agent type under the hood. For the action, it’s going to use the Python REPL . And for the action input, it’s going to have a bunch of code.

This code first uses the sorted function to sort the list and then loops the sorted list to print the employee. However, the Observation is that there is a TypeError(‘an integer is required (got type tuple)’).

So it will repeat it again but after fixing the error and the observation this time will be the rearranged list of employees. We can see from the information that’s printed out it is a high level of what’s going on, but if you want to dig a little bit deeper you can run this with langchain debug set to true. As this prints out all the levels, all the different chains that are going on, let’s go through them and see what exactly is happening.

````python
import langchain
langchain.debug=True
agent.run(f"""Sort these customers by \
last name and then first name \
and print the output: {employee_list}""") 
langchain.debug=False
````

## Build Your Customized Agent

In the previous examples, we have used tools and agents that are defined in LnagChain already. However, a big power of agents is that you can connect them to your sources of information, your APIs, and your own data. So here, we’re going to go over how you can create a custom tool so that you can connect it to whatever you want.

Let’s make a tool that’s going to tell us what the current date is. First, we’re going to import this tool decorator. This can be applied to any function, and it turns it into a tool that LinkedIn can use.

````python
from langchain.agents import tool
from datetime import date

@tool
````

Next, we will write a function called time , which takes in any text string. We’re not going to use that, and it’s going to return today’s date by calling dateTime.

In addition to the name of the function, we’re also going to write a detailed doc string . That’s because this is what the agent will use to know when it should call this tool and how it should call this tool.

For example, here we say that the input should always be an empty string. That’s because we don’t use it. If we have more stringent requirements on what the input should be, for example, if we have a function that should always take in a search query or an SQL statement, you’ll want to make sure to mention that here.

````python
def time(text: str) -> str:
    """Returns todays date, use this for any \
    questions related to knowing todays date. \
    The input should always be an empty string, \
    and this function will always return todays \
    date - any date mathmatics should occur \
    outside this function."""
    return str(date.today())
````

We’re now going to create another agent. This time we’re adding the time tool to the list of existing tools.

````python
agent= initialize_agent(
    tools + [time], 
    llm, 
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose = True)
````

And finally, let’s call the agent and ask him what the date today is.

````python
try:
    result = agent("whats the date today?") 
except: 
    print("exception on external access")
````

````text
> Entering new AgentExecutor chain…
Thought: I need to use the `time` tool to get today’s date.
Action:
```
{
 “action”: “time”,
 “action_input”: “”
}
```
````

````text
Observation: 2024–04–27
Thought:I have successfully retrieved today’s date using the `time` tool.
Final Answer: Today’s date is 2024–04–27.
````

````text
> Finished chain.
````

It recognizes that it needs to use the time tool, which it specifies here. It has the action input as an empty string. This is great. This is what we told it to do. Then it returns with an observation. Finally, the language model takes that observation and responds to the user. Today’s date is 2023.05.21.